# Bertolo, Weiss, Peretz & Sakata (2026) data

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

import sys
sys.path.append('../')
from src.data import save_units, load_units
from src.hmm import load_fit
from src.pipeline.fit_hmms import get_output_dir
from src.metrics.permutation import shuffle_labels_within_chunk

from dotenv import load_dotenv
load_dotenv()
if not os.getenv("BWPS_DIR"):
    raise Exception("BWPS_DIR environment variable not set")
DATA_DIR = Path(os.getenv("BWPS_DIR")) / "data"

ROOT_DIR = Path('.').resolve().parent
DATASET_ID = 'bwps' # Bertolo et al. 2026 dataset
OUTPUT_DIR = ROOT_DIR / 'data'

## Preprocess original dataset

We focus on the `birdcolony_IOI.csv` dataset, which contains "bird colony population, inter-onset intervals per bird ID". In `birdcolony_pairs.csv` we find the "mapping of which bird ID are related to each other (tutor-pupil relationships)".

Note that we do not look at `birdlong_IOI.csv` ("bird longitudinally recorded population, inter-onset intervals per bird ID")

In [2]:
bird_pairs = pd.read_csv(DATA_DIR / "birdcolony_pairs.csv")
bird_ids = set(bird_pairs['tutor'].tolist() + bird_pairs['pupil'].tolist())

In [3]:
df = pd.read_csv(DATA_DIR / "birdcolony_IOI.csv")

# Row 7159 duplicates the header, drop it
mask = (df['ID'] == 'ID') & (df['file'] == 'file') & (df['IOI'] == 'IOI')
df = df[~mask].reset_index(drop=True)

In [4]:
import pandas as pd

FILENAME_PATTERN = (
    r'(?P<bird_id>[a-zA-Z0-9]+)_(?:(?P<date>[A-Za-z]+_\d{1,2}_\d{4})_)?(?P<code>\d+)\.wav\.not\.mat$'
)

def parse_filenames(filenames):
    """Bird ID, date and code of each filename (NaN where a name does not match)."""
    parsed = filenames.str.extract(FILENAME_PATTERN)
    for filename in filenames[parsed['bird_id'].isna()]:
        print('NO MATCH:', filename)
    parsed['date'] = pd.to_datetime(parsed['date'].str.replace('_', ' '), errors='coerce')
    return parsed

In [5]:
# Note that some of the id's in the filenames do not match the ID's in the CSV file. 
processed_df = pd.concat([df, parse_filenames(df['file'])], axis=1)
mismatched_ids = processed_df[processed_df['bird_id'] != processed_df['ID']]
mismatched_ids = mismatched_ids[['ID', 'bird_id']].drop_duplicates()

for _, (id, bird_id) in mismatched_ids.iterrows():
    print(f"ID mismatch: '{id}' != '{bird_id}'")
    if id not in bird_ids:
        print(f" > ID {id} not in bird_ids")
    if bird_id not in bird_ids:
        print(f" > bird_id {bird_id} not in bird_ids")

ID mismatch: 'ixea_qwer' != 'ixea'
 > ID ixea_qwer not in bird_ids
ID mismatch: 'ye71ye72_2023_05_03' != 'ye71ye72'
 > ID ye71ye72_2023_05_03 not in bird_ids
ID mismatch: 'ye93ye94_2023_05_12' != 'ye93ye94'
 > ID ye93ye94_2023_05_12 not in bird_ids
ID mismatch: 'bl71bl81' != 'bl71bl181'
 > bird_id bl71bl181 not in bird_ids
ID mismatch: 'phpd_zxcv' != 'phpd'
 > ID phpd_zxcv not in bird_ids


Some of the id's in the filename do not match the ID's in the CSV file. Mostly it's the `ID`'s that are not in the birdcolony_pairs file, while the `bird_id`'s are. For example, ID `phpd_zxcv` does not appear in `birdcolony_pairs.csv`; `phpd` does, but `zxcv` does not. Since `phpd` does not appear in the IOIs data other than in `phpd_zxcv`, we want to replace `phpd_zxcv` by `phpd`. This is precisely what bird_id is already, so no need for further action.

There's one exception: bird_id `bl71bl181` probably contains a typo, an extra `1`, and should be `bl71bl81`. So we will drop `ID`, use `bird_id` instead, but replace `bl71bl181` by `bl71bl81`



We package all these preprocessing step into one function.

In [6]:
def extract_units(df):
    processed_df = pd.concat([df, parse_filenames(df['file'])], axis=1).drop(columns=['file'])

    # Fix Ids    
    processed_df.drop(columns=['ID'], inplace=True)
    processed_df.loc[processed_df['bird_id'] == 'bl71bl181', 'bird_id'] = 'bl71bl81'
    assert 'bl71bl181' not in processed_df['bird_id'].values

    # Rename IOI column and parse as floats
    processed_df.rename(columns=dict(IOI='ioi'), inplace=True)
    processed_df['ioi'] = processed_df['ioi'].astype(float)

    # Extract units
    prev_code = None
    prev_bird = None
    units = []
    start = 0
    chunk_idx = 0
    for interval, bird_id, date, code in processed_df.itertuples(index=False):
        if bird_id != prev_bird:
            # New bird
            chunk_idx = 0
            prev_bird = bird_id
            prev_code = None

        if code != prev_code:
            # New chunk
            start = 0
            chunk_idx += 1
            prev_code = code

        units.append(dict(
            start = start / 1000,
            duration = interval / 1000,
            chunk_idx=chunk_idx,
            bird_id=f"{DATASET_ID}-{bird_id}",
            date=date,
            code=code,
        ))
        start += interval

    columns=['start', 'duration', 'label', 'chunk_idx', 'bird_id', 'date', 'code']
    units = pd.DataFrame(units, columns=columns)
    units["label"] = units["label"].astype(str)
    return units


In [7]:
# Store units (v1)
units = extract_units(df)

from src.rhythm_types import drop_short_chunks

# A chunk needs at least 2 intervals to derive any dyad from it later.
n_before = units.groupby(['bird_id', 'chunk_idx']).ngroups
units = drop_short_chunks(units)
n_after = units.groupby(['bird_id', 'chunk_idx']).ngroups
print(f"dropped {n_before - n_after} of {n_before} chunks with fewer than 2 intervals")

save_units(units, 'syll_ioi', DATASET_ID, OUTPUT_DIR)

dropped 2 of 910 chunks with fewer than 2 intervals


## Augment

In [8]:
from src.data import load_units
from src.pipeline.predict_bird_ids import predict_bird_ids
from src.viz.changepoints import plot_changepoints

from src.pipeline.fit_hmms import get_output_dir
from src.hmm import load_fit

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



Next, we add automatically classified labels to each of the units. Note that this step is only possible after the units have been stored (above), and then hmms have been fitted for all birds using 

```sh
poetry run python -m src.pipeline.fit_hmms
```

The fitted models are written to disk, loaded and the predicted labels are stored as `label_pred`

In [9]:
# Add automatic labels, which requires running src.pipeline.fit_hmms
units = load_units(DATASET_ID, 'syll_ioi', use_milliseconds=False)

# Bird id predictions
result = predict_bird_ids(units, half_window=10, min_gap=5, id_prefix=f'{DATASET_ID}-pred-')
units['bird_id_pred'] = result['bird_id_pred']

# Add label predictions 
for bird_id, bird_group in units.groupby('bird_id'):
    path = get_output_dir(DATASET_ID, 'syll_ioi') / f"{bird_id}.joblib"
    result = load_fit(path)
    # The fit lists the chunks in chunk order, so the units must be too.
    assert bird_group['chunk_idx'].is_monotonic_increasing
    units.loc[bird_group.index, 'label_pred'] = np.concatenate(result['labels'])

# Shuffled baseline for plots (see preprocess-ns), shuffled from 'label_pred'
# since this dataset has no annotated labels.
rng = np.random.default_rng(0)
units['label_shuffle'] = shuffle_labels_within_chunk(
    units, rng, by=['bird_id', 'chunk_idx'], label_col='label_pred',
)['label_pred']

# Store units
save_units(units, 'syll_ioi', DATASET_ID, OUTPUT_DIR)